# CNN on the Emotions feature DataFrame — with TESS test

Train a CNN on the **tabular feature DataFrame** built from the `Emotions/` tracks (same features as
the *CNN_features* notebook). Each per-frame feature is summarized over time by **mean, std, min,
max**, so each clip is ~**101 features**; the **target is the `label` column**.

To keep the TESS test leakage-free, the `Emotions/` folder's TESS clips (`OAF_`/`YAF_` files) are
**excluded from training** and used only as the held-out **test set** (OAF and YAF separately).

Input is a flat feature vector, so we use a **1-D CNN over the feature axis**.

> **Setup:** `pip install librosa tensorflow scikit-learn pandas numpy matplotlib seaborn`

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import librosa
import matplotlib.pyplot as plt, seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")
np.random.seed(42); tf.random.set_seed(42)
print("librosa", librosa.__version__, "| tf", tf.__version__)

## 1. Configuration

In [ ]:
DATA_DIR = Path("../Emotions")
CSV_PATH = Path("emotions_features.csv")   # cached feature table (built if missing)
SR       = 22050
N_MFCC   = 13

SAMPLES_PER_CLASS = None     # None = every clip; set an int to sample fewer per emotion
PATIENCE = 8
EPOCHS   = 80
BATCH    = 32

## 2. Load or build the feature DataFrame

If `emotions_features.csv` exists (e.g. saved from the *CNN_features* notebook) it's loaded;
otherwise it's built here with the same librosa extractor. The TESS clips live inside `Emotions/`
as `OAF_`/`YAF_` files, so they're already in this table.

In [ ]:
def _stats(name, arr):
    """Four time-summary statistics for one per-frame feature row."""
    return {
        f"{name}_mean": float(np.mean(arr)),
        f"{name}_std":  float(np.std(arr)),
        f"{name}_min":  float(np.min(arr)),
        f"{name}_max":  float(np.max(arr)),
    }

def extract_features(path, sr=SR, n_mfcc=N_MFCC):
    """Per-clip librosa features summarized over time as mean/std/min/max."""
    y, sr = librosa.load(path, sr=sr)
    feats = {}

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        feats.update(_stats(f"mfcc{i+1}", mfcc[i]))

    feats.update(_stats("zcr",       librosa.feature.zero_crossing_rate(y)[0]))
    feats.update(_stats("rms",       librosa.feature.rms(y=y)[0]))
    feats.update(_stats("centroid",  librosa.feature.spectral_centroid(y=y, sr=sr)[0]))
    feats.update(_stats("bandwidth", librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]))
    feats.update(_stats("rolloff",   librosa.feature.spectral_rolloff(y=y, sr=sr)[0]))

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        feats.update(_stats(f"contrast{i+1}", contrast[i]))

    feats["duration"] = float(librosa.get_duration(y=y, sr=sr))
    return feats

## 3. Features (X), target (y), and TESS flag

Target is the `label` column. Flag TESS rows by filename (`OAF_`/`YAF_`) so we can keep them out of
training and use them as the test set.

In [ ]:
feature_cols = [c for c in features_df.columns if c not in ("file", "label")]
features_df["is_tess"] = features_df["file"].str.upper().str.startswith(("OAF_", "YAF_"))

le = LabelEncoder().fit(features_df["label"])     # fit on all 7 labels
n_classes = len(le.classes_)

print(f"{len(feature_cols)} features | classes: {list(le.classes_)}")
print(f"Non-TESS (train pool): {(~features_df['is_tess']).sum()}   TESS (test): {features_df['is_tess'].sum()}")

## 4. Split, scale, reshape

Train/internal-test split comes from the **non-TESS** rows. Standardize (fit on train), then add a
channel axis for the 1-D CNN: `(n_samples, n_features, 1)`.

In [ ]:
train_pool = features_df[~features_df["is_tess"]]
X = train_pool[feature_cols].astype("float32").values
y = le.transform(train_pool["label"])

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)[..., np.newaxis]
X_te = scaler.transform(X_te)[..., np.newaxis]

class_weight = dict(enumerate(
    compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)))
print("Train:", X_tr.shape, " Internal test:", X_te.shape)

## 5. Build the 1-D CNN

In [ ]:
def build_model(input_shape, n_classes):
    model = keras.Sequential([
        keras.layers.Input(shape=input_shape),
        layers.Conv1D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling1D(2), layers.Dropout(0.3),
        layers.Conv1D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation="relu"), layers.Dropout(0.4),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_model(X_tr.shape[1:], n_classes)
model.summary()

## 6. Train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=PATIENCE,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
]
history = model.fit(X_tr, y_tr, validation_split=0.15,
                    epochs=EPOCHS, batch_size=BATCH,
                    class_weight=class_weight, callbacks=callbacks, verbose=2)

h = history.history
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(h["accuracy"], label="train"); a1.plot(h["val_accuracy"], label="val")
a1.set_title("Accuracy"); a1.set_xlabel("Epoch"); a1.legend()
a2.plot(h["loss"], label="train"); a2.plot(h["val_loss"], label="val")
a2.set_title("Loss"); a2.set_xlabel("Epoch"); a2.legend()
plt.tight_layout(); plt.show()

## 7. Internal test (held-out non-TESS Emotions clips)

In [ ]:
test_acc = model.evaluate(X_te, y_te, verbose=0)[1]
print(f"Internal test accuracy: {test_acc:.3f}\n")
y_pred = model.predict(X_te, verbose=0).argmax(axis=1)
print(classification_report(y_te, y_pred, target_names=le.classes_, digits=3))

## 8. Test against the TESS folder — OAF vs YAF separately

The TESS rows (excluded from training) are the held-out cross-dataset test. Features are reused from
the DataFrame, scaled with the **same scaler**, and reported per speaker with a grouped bar chart.

In [ ]:
tess_df = features_df[features_df["is_tess"]]

def evaluate_tess(prefix, ax):
    rows = tess_df[tess_df["file"].str.upper().str.startswith(prefix + "_")]
    Xs = scaler.transform(rows[feature_cols].astype("float32").values)[..., np.newaxis]
    y_true = rows["label"].values
    y_pred = le.inverse_transform(model.predict(Xs, verbose=0).argmax(axis=1))

    acc = accuracy_score(y_true, y_pred)
    rep = classification_report(y_true, y_pred, labels=list(le.classes_),
                                target_names=list(le.classes_),
                                output_dict=True, zero_division=0)
    print(f"\n===== {prefix}  (n={len(rows)})   accuracy: {acc:.3f} =====")
    print(classification_report(y_true, y_pred, labels=list(le.classes_),
                                target_names=list(le.classes_), digits=3, zero_division=0))

    classes = list(le.classes_); metrics = ["precision", "recall", "f1-score"]
    x = np.arange(len(classes)); w = 0.25
    for k, mtr in enumerate(metrics):
        ax.bar(x + (k - 1) * w, [rep[c][mtr] for c in classes], w, label=mtr)
    ax.axhline(acc, color="black", ls="--", lw=1, label=f"accuracy={acc:.2f}")
    ax.set_xticks(x); ax.set_xticklabels(classes, rotation=30, ha="right")
    ax.set_ylim(0, 1.05); ax.set_title(f"{prefix} - TESS (n={len(rows)})")
    ax.legend(fontsize=8, ncol=2)
    return acc

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
acc_oaf = evaluate_tess("OAF", axes[0])
acc_yaf = evaluate_tess("YAF", axes[1])
plt.suptitle(f"TESS performance  |  OAF acc={acc_oaf:.3f}   YAF acc={acc_yaf:.3f}", y=1.02)
plt.tight_layout(); plt.show()